# Phys-gated Morgan quantile mixture of experts

This notebook fits `PhysGatedMorganQuantileMoE` (see `src/models/moe_quantile.py`): a **Gaussian mixture** on the first three RDKit physicochemical descriptors (**MolWt**, **MolLogP**, **TPSA**) gates **K** experts; each expert is an `EnsembleQuantileRegressor` over several **HistGradientBoosting** quantile models on **`morgan_r1_count_1024`**.

**Plots:** test-set observed vs median prediction with quantile bands, empirical interval coverage, gate-space scatter colored by dominant mixture component, and interval-width distribution.

Run from the repo root or `notebooks/`; the first code cell adds `src/` to `sys.path`. `load_data.train` may download the training CSV from Hugging Face on first import.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from rdkit import Chem
from sklearn.model_selection import train_test_split

_REPO = Path.cwd().resolve()
if not (_REPO / "src" / "baseline.py").exists():
    _REPO = _REPO.parent
sys.path.insert(0, str(_REPO / "src"))

from baseline import BaselineCVConfig, prepare_training_data
from features_data import build_descriptor_matrix, build_rdkit_phys_props_matrix
from load_data import train
from models import PhysGatedMorganQuantileMoE

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110

RANDOM_STATE = 0
Y_COL = "pEC50"
DESCRIPTOR = "morgan_r1_count_1024"
GATE_NAMES = ("MolWt", "MolLogP", "TPSA")

## Load aligned molecules, Morgan FP, and gate features

`X_gate` uses columns 0–2 of `build_rdkit_phys_props_matrix` (same order as `RDKIT_PHYS_PROP_NAMES` in `features_data.py`).

In [ ]:
cfg = BaselineCVConfig(y_col=Y_COL)
mols = list(train["SMILES"].apply(Chem.MolFromSmiles))
y, mols_f, mask = prepare_training_data(train, mols, y_col=cfg.y_col)

X_phys = build_rdkit_phys_props_matrix(mols_f).astype(np.float64)
X_gate = X_phys[:, :3]
X_morgan = build_descriptor_matrix(DESCRIPTOR, mols_f).astype(np.float64)

print(
    f"n={len(y)}  {DESCRIPTOR} {X_morgan.shape}  gate {X_gate.shape}  "
    f"finite y: {np.isfinite(y).all()}"
)

## Train / test split and minimal `dataset` wrapper

`PhysGatedMorganQuantileMoE` only needs `.y` and `len(dataset)` for row alignment; features are passed as `X_gate` / `X_morgan`.

In [ ]:
class YOnlyDataset:
    def __init__(self, y: np.ndarray) -> None:
        yy = np.asarray(y, dtype=np.float64)
        self.y = yy.reshape(-1, 1) if yy.ndim == 1 else yy

    def __len__(self) -> int:
        return int(self.y.shape[0])


Xm_tr, Xm_te, Xg_tr, Xg_te, y_tr, y_te = train_test_split(
    X_morgan,
    X_gate,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    shuffle=True,
)
ds_tr = YOnlyDataset(y_tr)
ds_te = YOnlyDataset(y_te)
print(f"train {len(ds_tr)}  test {len(ds_te)}")

## Fit MoE

Smaller `max_iter` keeps the notebook responsive; increase for tighter fits. **`n_components_gmm`** is the number of mixture components (experts).

In [ ]:
quantile_levels = (0.1, 0.5, 0.9)
moe = PhysGatedMorganQuantileMoE(
    n_components_gmm=4,
    quantile_levels=quantile_levels,
    n_ensemble_members=2,
    gmm_kwargs={"covariance_type": "diag", "n_init": 3},
    hgb_kwargs={"max_iter": 120, "max_depth": 6},
    random_state=RANDOM_STATE,
)
moe.fit(ds_tr, X_gate=Xg_tr, X_morgan=Xm_tr)
print("Fitted MoE.")

## Predictions and metrics on the test set

In [ ]:
q_te = moe.predict_quantiles(ds_te, X_gate=Xg_te, X_morgan=Xm_te)
i_lo, i_med, i_hi = 0, 1, 2
q_lo = q_te[:, 0, i_lo]
q_med = q_te[:, 0, i_med]
q_hi = q_te[:, 0, i_hi]
y_flat = y_te.ravel()

rmse = float(np.sqrt(np.mean((q_med - y_flat) ** 2)))
mae = float(np.mean(np.abs(q_med - y_flat)))
pinball = moe.evaluate_pinball_loss(ds_te, X_gate=Xg_te, X_morgan=Xm_te)
cover = float(np.mean((y_flat >= q_lo) & (y_flat <= q_hi)))
nominal = quantile_levels[i_hi] - quantile_levels[i_lo]
width = q_hi - q_lo

print(f"Test RMSE (median): {rmse:.4f}")
print(f"Test MAE (median):  {mae:.4f}")
print(f"Mean pinball (over quantile levels): {pinball:.4f}")
print(f"Empirical coverage [{quantile_levels[i_lo]}, {quantile_levels[i_hi]}]: {cover:.3f}  (nominal ~{nominal:.2f})")
print(f"Mean interval width: {width.mean():.4f}")

## Plot 1 — Observed vs median with quantile intervals

Each point shows the **0.1–0.9** quantile range as a vertical segment (not calibrated conformal intervals).

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
lo, hi = float(y_flat.min()), float(y_flat.max())
pad = 0.5 * (hi - lo)
lims = (lo - pad, hi + pad)

for yi, lo_i, hi_i, med_i in zip(y_flat, q_lo, q_hi, q_med, strict=True):
    ax.vlines(yi, lo_i, hi_i, colors="C0", alpha=0.12, linewidth=1)
ax.scatter(y_flat, q_med, s=12, alpha=0.5, c="C0", edgecolors="none", label="median")
ax.plot(lims, lims, "k--", linewidth=1, label="identity")
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel(f"Observed {Y_COL}")
ax.set_ylabel("Predicted (MoE median)")
ax.set_title("Test set: median + 10–90% quantile segments")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

## Plot 2 — Gate space (MolWt vs MolLogP) colored by dominant expert

Dominant component = argmax of GMM **posterior** on scaled/imputed gate features.

In [ ]:
Z_te = moe._gate_pre.transform(Xg_te)
proba = moe._gmm.predict_proba(Z_te)
dominant = np.argmax(proba, axis=1)

fig, ax = plt.subplots(figsize=(7, 5.5))
sc = ax.scatter(
    Xg_te[:, 0],
    Xg_te[:, 1],
    c=dominant,
    cmap="tab10",
    alpha=0.65,
    s=18,
)
cb = plt.colorbar(sc, ax=ax, label="mixture component")
ax.set_xlabel(GATE_NAMES[0])
ax.set_ylabel(GATE_NAMES[1])
ax.set_title("Test molecules: gate features (raw) by dominant GMM component")
plt.tight_layout()
plt.show()

## Plot 3 — Distribution of predicted interval widths

Width = $\hat{q}_{0.9} - \hat{q}_{0.1}$ per compound.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.histplot(width, kde=True, ax=ax, color="C2")
ax.axvline(float(np.median(width)), color="k", linestyle="--", label=f"median = {np.median(width):.3f}")
ax.set_xlabel("Predicted 90% − 10% quantile width")
ax.set_ylabel("Count")
ax.set_title("Test set interval widths (MoE)")
ax.legend()
plt.tight_layout()
plt.show()